# ISLR Ch. 3 - Q15 (Boston)

This notebook downloads the Boston Housing dataset and saves it as `boston.csv` in the project root. Run the first cell to fetch the data; subsequent analysis can read from that CSV.

In [ ]:
# Download Boston Housing data and save as boston.csv
# Tries a GitHub CSV first, then falls back to OpenML via scikit-learn.

from __future__ import annotations
import io
import os
import pathlib
import urllib.request
import pandas as pd

def download_boston_csv(out_path: str = 'boston.csv') -> pd.DataFrame:
    """Download the Boston Housing dataset and save as CSV.
    Attempts GitHub-hosted CSV first, then OpenML (via scikit-learn).

    Returns the loaded DataFrame.
    """
    out_path = str(pathlib.Path(out_path))
    if os.path.exists(out_path):
        df = pd.read_csv(out_path)
        print(f'Found existing {out_path} - shape: {df.shape}')
        return df

    github_urls = [
        'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv',
        'https://raw.githubusercontent.com/khaki3/BostonHousing/master/BostonHousing.csv',
    ]

    for url in github_urls:
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                data = r.read()
            df = pd.read_csv(io.BytesIO(data))
            # Normalize column names for consistency with ISLR (lowercase)
            df.columns = [c.lower() for c in df.columns]
            df.to_csv(out_path, index=False)
            print(f'Downloaded from {url} -> {out_path} - shape: {df.shape}')
            return df
        except Exception as e:
            print(f'GitHub source failed: {url} - {e}')

    # Fallback: OpenML via scikit-learn (may show deprecation warning)
    try:
        from sklearn.datasets import fetch_openml
        boston = fetch_openml(name='boston', version=1, as_frame=True)
        df = pd.concat([boston.data, boston.target.rename('medv')], axis=1)
        df.columns = [c.lower() for c in df.columns]
        df.to_csv(out_path, index=False)
        print(f'Fetched from OpenML -> {out_path} - shape: {df.shape}')
        return df
    except Exception as e:
        raise RuntimeError('All download methods failed; check your internet connection.') from e

# Execute download and preview
df = download_boston_csv('boston.csv')
display(df.head())
assert 'crim' in df.columns, "Expected 'crim' in dataset columns."
print('Columns:', ', '.join(df.columns))


## Data Setup
Loads `boston.csv` created above and prepares predictors (`X`) and response (`y = crim`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

pd.options.display.float_format = '{:.3f}'.format
sns.set_theme(style='whitegrid', context='notebook')

df = pd.read_csv('boston.csv')
df.columns = [c.lower() for c in df.columns]
# Some sources name the 'B' variable as 'black' or 'b'. Keep whichever exists.
if 'black' in df.columns and 'b' not in df.columns:
    df = df.rename(columns={'black': 'b'})

target = 'crim'
predictors = [c for c in df.columns if c != target]
X = df[predictors].copy()
y = df[target].copy()
df.shape, target, predictors[:5]


## (a) Univariate Regressions
Fit `crim ~ X_j` for each predictor. Report slope and p-value, and show quick plots.

In [ ]:
uni_results = []
for col in predictors:
    Xi = sm.add_constant(X[[col]])
    model = sm.OLS(y, Xi, missing='drop').fit()
    slope = model.params[col]
    pval = model.pvalues[col]
    r2 = model.rsquared
    uni_results.append({'predictor': col, 'slope': slope, 'p_value': pval, 'r2': r2})

uni_df = pd.DataFrame(uni_results).sort_values('p_value')
display(uni_df)

# Plot: grid of scatter + OLS line
n = len(predictors)
cols = 4
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.2*rows), squeeze=False)
for i, col in enumerate(predictors):
    r, c = divmod(i, cols)
    ax = axes[r, c]
    sns.regplot(x=df[col], y=y, ax=ax, scatter_kws={'s': 12, 'alpha': 0.6}, line_kws={'color':'tab:red'})
    ax.set_xlabel(col)
    ax.set_ylabel('crim')
# hide any empty subplots
for j in range(n, rows*cols):
    r, c = divmod(j, cols)
    axes[r, c].axis('off')
fig.suptitle('Univariate relationships: crim vs predictor', y=1.02)
plt.tight_layout()
plt.show()


## (b) Multiple Regression
Fit `crim ~` all predictors. Identify predictors with significant coefficients (p < 0.05).

In [ ]:
X_all = sm.add_constant(X)
multi_model = sm.OLS(y, X_all, missing='drop').fit()
display(multi_model.summary())
sig_multi = (multi_model.pvalues.drop('const') < 0.05)
sig_predictors = list(sig_multi[sig_multi].index)
print('Significant at 5%:', sig_predictors)


## (c) Compare Coefficients
Scatter of univariate slopes (x-axis) vs multiple-regression coefficients (y-axis).

In [ ]:
coef_uni = uni_df.set_index('predictor')['slope']
coef_multi = multi_model.params.drop('const')
cmp = pd.DataFrame({'uni_slope': coef_uni, 'multi_coef': coef_multi})
display(cmp)

plt.figure(figsize=(6.5,6))
sns.scatterplot(x='uni_slope', y='multi_coef', data=cmp.reset_index())
lims = [cmp.min().min(), cmp.max().max()]
m = max(abs(lims[0]), abs(lims[1]))
plt.plot([-m, m], [-m, m], 'k--', linewidth=1)
for name, row in cmp.iterrows():
    plt.annotate(name, (row['uni_slope'], row['multi_coef']), fontsize=8, alpha=0.8)
plt.xlabel('Univariate slope')
plt.ylabel('Multiple-regression coefficient')
plt.title('Coefficient comparison (expect shrinkage and sign changes)')
plt.grid(True, alpha=0.3)
plt.show()


## (d) Evidence of Nonlinearity?
For each predictor `X`, compare linear model to cubic (`X, X^2, X^3`) using an F-test. Report p-values and show top few curves.

In [ ]:
from collections import OrderedDict

nonlin = []
for col in predictors:
    x1 = X[[col]].astype(float)
    X_lin = sm.add_constant(x1)
    mdl_lin = sm.OLS(y, X_lin, missing='drop').fit()

    X_poly = x1.copy()
    X_poly[col + '^2'] = x1[col]**2
    X_poly[col + '^3'] = x1[col]**3
    X_poly = sm.add_constant(X_poly)
    mdl_cub = sm.OLS(y, X_poly, missing='drop').fit()

    F, p, df_diff = mdl_cub.compare_f_test(mdl_lin)
    nonlin.append({'predictor': col, 'F': F, 'p_value': p, 'df_diff': df_diff})

nonlin_df = pd.DataFrame(nonlin).sort_values('p_value')
display(nonlin_df)
print('Predictors showing nonlinearity at 5%:', list(nonlin_df.loc[nonlin_df.p_value < 0.05, 'predictor']))

# Plot fitted cubic curves for the top 4 strongest nonlinearities
top = list(nonlin_df.head(4)['predictor'])
fig, axes = plt.subplots(2, 2, figsize=(10,8))
for ax, col in zip(axes.ravel(), top):
    x = X[col].astype(float)
    xs = np.linspace(x.min(), x.max(), 200)
    X_poly = sm.add_constant(pd.DataFrame({col: xs, col+'^2': xs**2, col+'^3': xs**3}))
    mdl = sm.OLS(y, sm.add_constant(pd.DataFrame({col: x, col+'^2': x**2, col+'^3': x**3})), missing='drop').fit()
    yhat = mdl.predict(X_poly)
    ax.scatter(x, y, s=10, alpha=0.5)
    ax.plot(xs, yhat, color='tab:red', lw=2)
    ax.set_xlabel(col)
    ax.set_ylabel('crim')
    ax.set_title(f'Cubic fit: {col}')
plt.tight_layout()
plt.show()
